# Phase III – Part A: Faber Baseline Strategy
Implements Faber's 10-month SMA momentum rule:
- **Buy** when month-end price > 10-month SMA
- **Move to cash** when month-end price < 10-month SMA
- Equal weight across all invested ETFs
- Monthly rebalancing from February 2007 to December 2024

In [24]:
import pandas as pd
import numpy as np

ETFS       = ['SPY', 'EFA', 'TLT', 'VNQ', 'DBC']
BACKTEST_START = '2010-09-30'
BACKTEST_END   = '2024-08-31'

prices = pd.read_csv('cleaned_prices.csv', index_col=0, parse_dates=True)
print(f'Loaded {len(prices)} month-end observations.')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')

Loaded 227 month-end observations.
Date range: 2006-02-28 to 2024-12-31


In [30]:
tbillrates = pd.read_csv('tbillrates.csv', index_col=0, parse_dates=True)

In [31]:
# 1. Create a range that explicitly goes up to your backtest end date
# This ensures 2024-08-31 exists in the index even if the CSV ends early
all_days = pd.date_range(start=tbillrates.index.min(), end=BACKTEST_END, freq='D')

# 2. Reindex to this extended range and fill missing values forward
tbillrates = tbillrates.reindex(all_days).ffill()

# 3. Standardize name
tbillrates.index.name = 'observation_date'

tbillrates now ends on: 2024-08-31


In [33]:
tbillrates

,DTB3_20260430
observation_date,
2010-09-30,0.16
2010-10-01,0.16
2010-10-02,0.16
2010-10-03,0.16
2010-10-04,0.13
...,...
2024-08-27,4.98
2024-08-28,4.96
2024-08-29,4.98


## Step 1: Calculate 10-Month SMA
For each ETF at each month-end, compute the simple moving average of the last 10 monthly closing prices.

In [3]:
# rolling(10) uses only past prices — no lookahead bias
sma_10 = prices[ETFS].rolling(10).mean()

print('10-month SMA calculated.')
print(f'First valid SMA date: {sma_10.dropna().index[0].date()}')

10-month SMA calculated.
First valid SMA date: 2006-11-30


## Step 2: Generate Buy/Sell Signals
Signal = 1 (invest) if price > SMA, else 0 (cash).

In [4]:
signals = (prices[ETFS] > sma_10).astype(int)

# Restrict to backtest period
signals = signals.loc[BACKTEST_START:BACKTEST_END]
prices_bt = prices[ETFS].loc[BACKTEST_START:BACKTEST_END]

print(f'Backtest period: {signals.index[0].date()} to {signals.index[-1].date()}')
print(f'{len(signals)} monthly observations.')
signals.head()

Backtest period: 2010-09-30 to 2024-08-31
168 monthly observations.


,SPY,EFA,TLT,VNQ,DBC
2010-09-30,1,1,1,1,1
2010-10-31,1,1,1,1,1
2010-11-30,1,1,1,1,1
2010-12-31,1,1,0,1,1
2011-01-31,1,1,0,1,1


## Step 3: Run Monthly Rebalancing Loop
At each month-end:
- Identify which ETFs have price > 10-month SMA
- Assign equal weight to each invested ETF
- If no ETFs qualify, portfolio sits in cash (0% return)
- Record the weighted portfolio return for that month

In [15]:
signals

,SPY,EFA,TLT,VNQ,DBC
2010-09-30,1,1,1,1,1
2010-10-31,1,1,1,1,1
2010-11-30,1,1,1,1,1
2010-12-31,1,1,0,1,1
2011-01-31,1,1,0,1,1
...,...,...,...,...,...
2024-04-30,1,1,0,0,1
2024-05-31,1,1,0,1,1
2024-06-30,1,1,1,1,1
2024-07-31,1,1,1,1,0


In [16]:
portfolio_returns

[{'date': Timestamp('2010-09-30 00:00:00'),
  'return': 0.058962873890550424,
  'n_invested': 5},
 {'date': Timestamp('2010-10-31 00:00:00'),
  'return': 0.02467420246533685,
  'n_invested': 5},
 {'date': Timestamp('2010-11-30 00:00:00'),
  'return': -0.017602789755462145,
  'n_invested': 5},
 {'date': Timestamp('2010-12-31 00:00:00'),
  'return': 0.07359079962336051,
  'n_invested': 4},
 {'date': Timestamp('2011-01-31 00:00:00'),
  'return': 0.02808398266935208,
  'n_invested': 4},
 {'date': Timestamp('2011-02-28 00:00:00'),
  'return': 0.03974942905716461,
  'n_invested': 4},
 {'date': Timestamp('2011-03-31 00:00:00'),
  'return': -0.003321904727117242,
  'n_invested': 4},
 {'date': Timestamp('2011-04-30 00:00:00'),
  'return': 0.04706096584957303,
  'n_invested': 4},
 {'date': Timestamp('2011-05-31 00:00:00'),
  'return': -0.007404772660739091,
  'n_invested': 5},
 {'date': Timestamp('2011-06-30 00:00:00'),
  'return': -0.02615356263748264,
  'n_invested': 4},
 {'date': Timestamp('2

In [17]:
monthly_returns

,SPY,EFA,TLT,VNQ,DBC
2010-09-30,0.089555,0.099720,-0.025204,0.044707,0.086036
2010-10-31,0.038202,0.038055,-0.044693,0.047427,0.044380
2010-11-30,0.000000,-0.048237,-0.016893,-0.018515,-0.004369
2010-12-31,0.066852,0.083081,-0.036990,0.045507,0.098923
2011-01-31,0.023300,0.020955,-0.030812,0.032509,0.035572
...,...,...,...,...,...
2024-04-30,-0.040320,-0.032432,-0.064554,-0.079441,0.016108
2024-05-31,0.050580,0.050602,0.028870,0.045597,-0.002999
2024-06-30,0.035280,-0.018267,0.018171,0.018756,-0.001719
2024-07-31,0.012109,0.025916,0.036299,0.079393,-0.027981


In [36]:
monthly_returns = prices[ETFS].pct_change()
monthly_returns = monthly_returns.loc[BACKTEST_START:BACKTEST_END]

portfolio_returns = []

for date in signals.index:
    invested = signals.loc[date]
    n_invested = invested.sum()

    # Equal weight across invested ETFs
    weight = 0.2
    tbillrate = tbillrates.loc[date].iloc[0]/100/12
    ret = (monthly_returns.loc[date] * invested * weight).sum()+tbillrate*(5.0-n_invested)/5.0
    portfolio_returns.append({'date': date, 'return': ret, 'n_invested': int(n_invested)})

faber_returns = pd.DataFrame(portfolio_returns).set_index('date')

print(f'Backtest complete: {len(faber_returns)} monthly observations.')
print(f'Months fully in cash: {(faber_returns["n_invested"] == 0).sum()}')
faber_returns.head(10)

Backtest complete: 168 monthly observations.
Months fully in cash: 4


,return,n_invested
date,,
2010-09-30,0.058963,5
2010-10-31,0.024674,5
2010-11-30,-0.017603,5
2010-12-31,0.058893,4
2011-01-31,0.022492,4
2011-02-28,0.031825,4
2011-03-31,-0.002643,4
2011-04-30,0.037655,4
2011-05-31,-0.007405,5


## Step 4: Save Results

In [5]:
faber_returns.to_csv('faber_returns.csv')
print('Saved: faber_returns.csv')

Saved: faber_returns.csv
